# RSNA: CoAtNet + MedGemma soft-label studies, no disk image cache

Run the code cells in order. Attach RSNA competition data, the CoAtNet checkpoint, and the weak-label CSV configured below.

This version intentionally **does not cache preprocessed DICOM studies to `/kaggle/working`**. Each study is decoded and preprocessed on demand by the dataset/inference code. This trades extra CPU/DICOM decoding time for dramatically lower Kaggle disk usage. `/kaggle/working` is used only for model checkpoints, metric CSVs, series-selection CSVs, and `submission.csv`.

Gold loss remains 1.0; uncertain findings retain their configured lower weight and unmentioned findings weight 0.0. The final CoAtNet stage and classification/pooling heads are trained; earlier stages stay frozen.

Dependencies: torch, torchvision, timm, pandas, numpy, scikit-learn, opencv-python-headless, pydicom, safetensors. For compressed DICOMs install pylibjpeg, pylibjpeg-libjpeg and pylibjpeg-openjpeg if absent. Enable a Kaggle GPU.


In [1]:
from pathlib import Path
from contextlib import nullcontext
from concurrent.futures import ThreadPoolExecutor
import time, random, gc
import numpy as np
import pandas as pd
import torch, cv2, pydicom, timm
from torch import nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import KFold
from sklearn.metrics import roc_auc_score

SEED=42; random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.backends.cudnn.benchmark=True
LABEL_COLS=['ACL','MCL','Medial Meniscus','Lateral Meniscus','Medial OA','Lateral OA','PF OA','Effusion','Synovitis',"Baker's",'Contusion','Fracture']
IMAGE_SIZE=224; CENTERS_PER_PLANE=10; IMAGE_MICROBATCH=5
NUM_DICOM_THREADS=4; NUM_WORKERS=2; N_FOLDS=4; EPOCHS=8
PSEUDO_LOSS_WEIGHT=.75; EXPECTED_PSEUDO_COUNT=1750
RUN_TAG=f"coatnet_medgemma{EXPECTED_PSEUDO_COUNT}_w{int(round(PSEUDO_LOSS_WEIGHT*100)):02d}"
WEAK_LABEL_CSV= Path("/kaggle/input/datasets/amritgopinath/medgemma-27b-labels/rsna_medgemma_modal27b_soft1750.csv")
LR_BACKBONE=1e-4; LR_HEAD=5e-4
DATA_CANDIDATES=[Path('/kaggle/input/rsna-knee-abnormality-detection'),Path('/kaggle/input/competitions/rsna-knee-abnormality-detection')]
DATA_ROOT=next((p for p in DATA_CANDIDATES if p.exists()),None)
if DATA_ROOT is None: raise FileNotFoundError('Attach RSNA Knee Abnormality Detection competition data.')
TRAIN_SERIES_ROOT=DATA_ROOT/'train_series'; TEST_SERIES_ROOT=DATA_ROOT/'test_series'
OUT=Path('/kaggle/working')
COATNET_WEIGHTS= Path('/kaggle/input/models/amritgopinath/coatnet-fine-tuned/pytorch/default/1/coatnet_kneemri_backbone.pt')
print('device',DEVICE,'timm',timm.__version__)


device cuda timm 1.0.26


In [2]:
def find_coatnet():
    # Bounded-depth lookup; no recursive traversal of competition DICOM folders.
    root=Path('/kaggle/input'); matches=[]
    for name in ('model.safetensors','pytorch_model.bin','coatnet_0_rw_224.pth'):
        for pattern in (f'*/{name}',f'*/*/{name}',f'datasets/*/*/{name}',f'datasets/*/*/*/{name}'):
            matches.extend(p for p in root.glob(pattern) if any('coatnet' in x.lower() for x in p.parts))
    matches=sorted(set(matches))
    if len(matches)>1: raise ValueError(f'Multiple CoAtNet checkpoints: {matches}. Set COATNET_WEIGHTS explicitly.')
    return matches[0] if matches else None
COATNET_WEIGHTS=Path(COATNET_WEIGHTS) if COATNET_WEIGHTS else find_coatnet()
if COATNET_WEIGHTS is None or not COATNET_WEIGHTS.is_file():
    raise FileNotFoundError('Attach official timm coatnet_0_rw_224.sw_in1k weights and set COATNET_WEIGHTS.')
print('ImageNet weights',COATNET_WEIGHTS)


ImageNet weights /kaggle/input/models/amritgopinath/coatnet-fine-tuned/pytorch/default/1/coatnet_kneemri_backbone.pt


In [3]:
train=pd.read_csv(DATA_ROOT/'train.csv',dtype={'StudyInstanceUID':str})
train_series=pd.read_csv(DATA_ROOT/'train_series.csv',dtype={'StudyInstanceUID':str,'SeriesInstanceUID':str})
test_series=pd.read_csv(DATA_ROOT/'test_series.csv',dtype={'StudyInstanceUID':str,'SeriesInstanceUID':str})
sample_submission=pd.read_csv(DATA_ROOT/'sample_submission.csv',dtype={'StudyInstanceUID':str})
assert list(sample_submission.columns)==['StudyInstanceUID']+LABEL_COLS
gold=train[train[LABEL_COLS].notna().all(axis=1)].copy().reset_index(drop=True)
gold[LABEL_COLS]=gold[LABEL_COLS].apply(pd.to_numeric)
assert len(gold)==58, f'Expected 58 gold studies; got {len(gold)}'
assert gold.StudyInstanceUID.is_unique
print('gold studies',len(gold),'test studies',len(sample_submission))

# Load only the exact export name, never recursively scan attached inputs.
import hashlib
CSV_NAME=WEAK_LABEL_CSV.name if WEAK_LABEL_CSV is not None else 'rsna_medgemma_modal27b_soft1750.csv'
if WEAK_LABEL_CSV is not None:
    csv_paths=[Path(WEAK_LABEL_CSV)]
else:
    root=Path('/kaggle/input')
    csv_paths=sorted(set([p for pattern in (f'*/{CSV_NAME}',f'datasets/*/*/{CSV_NAME}')
                         for p in root.glob(pattern)] + ([OUT/CSV_NAME] if (OUT/CSV_NAME).is_file() else [])))
if len(csv_paths)!=1 or not csv_paths[0].is_file():
    raise FileNotFoundError(f'Set WEAK_LABEL_CSV to the {EXPECTED_PSEUDO_COUNT}-row {CSV_NAME}; found {csv_paths}')
print('Reading weak labels from',csv_paths[0])
pseudo=pd.read_csv(csv_paths[0],dtype={'StudyInstanceUID':str,'report_sha256':str})
assert len(pseudo)==EXPECTED_PSEUDO_COUNT and pseudo.StudyInstanceUID.is_unique, f'Expected {EXPECTED_PSEUDO_COUNT} unique rows; got {len(pseudo)}'
required={'StudyInstanceUID','report_sha256','model','prompt_version',*LABEL_COLS,
          *('weight__'+x for x in LABEL_COLS),*('status__'+x for x in LABEL_COLS)}
assert required.issubset(pseudo.columns), f'Missing CSV columns: {required-set(pseudo.columns)}'
assert 'Report' in train.columns and train.StudyInstanceUID.is_unique
assert set(pseudo.StudyInstanceUID).isdisjoint(set(gold.StudyInstanceUID)), 'Gold study in weak labels'
assert set(pseudo.StudyInstanceUID).issubset(set(train.StudyInstanceUID)), 'Unexpected study ID'
assert pseudo.model.eq('google/medgemma-27b-text-it').all()
assert pseudo.prompt_version.eq('medgemma-knee-v1-modal-text-v1').all()
report_lookup=train.set_index('StudyInstanceUID')['Report']
for row in pseudo[['StudyInstanceUID','report_sha256']].itertuples(index=False):
    digest=hashlib.sha256(str(report_lookup.loc[row.StudyInstanceUID]).encode('utf-8')).hexdigest()
    if row.report_sha256!=digest:
        raise ValueError(f'Report hash mismatch for study {row.StudyInstanceUID}')
soft_map={'positive':(.95,1.),'negative':(.05,1.),'uncertain':(.5,.2),'not_mentioned':(.5,0.)}
for col in LABEL_COLS:
    values=pd.to_numeric(pseudo[col],errors='raise')
    weights=pd.to_numeric(pseudo['weight__'+col],errors='raise')
    expected=pseudo['status__'+col].map(soft_map)
    if expected.isna().any(): raise ValueError(f'Invalid status for {col}')
    assert np.allclose(values.to_numpy(),expected.map(lambda v:v[0]).to_numpy())
    assert np.allclose(weights.to_numpy(),expected.map(lambda v:v[1]).to_numpy())
    pseudo[col]=values.astype(float)
    pseudo['weight__'+col]=weights.astype(float)
pseudo=pseudo.sample(frac=1,random_state=SEED).reset_index(drop=True)
print('report-only pseudo rows',len(pseudo),'usable targets',int(pseudo[['weight__'+x for x in LABEL_COLS]].gt(0).sum().sum()))


gold studies 58 test studies 3
Reading weak labels from /kaggle/input/datasets/amritgopinath/medgemma-27b-labels/rsna_medgemma_modal27b_soft1750.csv
report-only pseudo rows 1750 usable targets 14617


In [4]:
from safetensors.torch import load_file as load_safetensors

def make_backbone(load_pretrained=True):
    net=timm.create_model('coatnet_0_rw_224.sw_in1k',pretrained=False)
    expected_size=int(timm.data.resolve_model_data_config(net)['input_size'][-1])
    if IMAGE_SIZE != expected_size:
        raise ValueError(f'IMAGE_SIZE={IMAGE_SIZE} is incompatible with this CoAtNet checkpoint/model; expected {expected_size}.')
    net.reset_classifier(0)  # Match a feature-only or full ImageNet checkpoint.
    if load_pretrained and COATNET_WEIGHTS.suffix=='.safetensors': state=load_safetensors(str(COATNET_WEIGHTS),device='cpu')
    elif load_pretrained:
        state=torch.load(COATNET_WEIGHTS,map_location='cpu',weights_only=True)
        if isinstance(state,dict): state=state.get('state_dict',state.get('model',state))
        state={k.removeprefix('module.'):v for k,v in state.items()}
    if load_pretrained:
        # Classifier keys are optional and unused for the 12-label task.
        # Keep strict=True for every remaining backbone weight and shape.
        state={k.removeprefix('module.'):v for k,v in state.items()}
        for key in ('head.fc.weight','head.fc.bias'):
            state.pop(key,None)
        net.load_state_dict(state,strict=True)
    for p in net.parameters(): p.requires_grad_(False)
    if not hasattr(net,'stages'): raise RuntimeError('Unexpected timm CoAtNet layout; inspect net and choose its final stage.')
    for p in net.stages[-1].parameters(): p.requires_grad_(True)
    return net

class StudyCoAtNet(nn.Module):
    def __init__(self,load_pretrained=True):
        super().__init__()
        self.backbone=make_backbone(load_pretrained)
        dim=self.backbone.num_features
        self.head=nn.Sequential(nn.LayerNorm(dim),nn.Dropout(.15),nn.Linear(dim,len(LABEL_COLS)))
        self.plane_weight=nn.Parameter(torch.zeros(4,len(LABEL_COLS)))
        self.register_buffer('mean',torch.tensor([.485,.456,.406]).view(1,3,1,1))
        self.register_buffer('std',torch.tensor([.229,.224,.225]).view(1,3,1,1))

    def train_backbone_tail(self):
        self.backbone.eval()
        self.backbone.stages[-1].train()

    def forward(self,images,planes):
        slice_logits=[]
        for chunk in images.split(IMAGE_MICROBATCH):
            x=(chunk.to(dtype=self.mean.dtype)/255-self.mean)/self.std
            ctx=torch.autocast('cuda',dtype=torch.float16) if self.training and images.is_cuda else nullcontext()
            with ctx: slice_logits.append(self.head(self.backbone(x)))
        scores=torch.cat(slice_logits,0)
        pooled=[]; available=[]
        for plane in range(4):
            part=scores[planes==plane]
            available.append(len(part)>0)
            pooled.append(part.topk(min(2,len(part)),dim=0).values.mean(0) if len(part) else torch.zeros_like(scores[0]))
        mask=torch.tensor(available,device=images.device)
        weights=self.plane_weight.masked_fill(~mask[:,None],-1e4).softmax(0)
        return (weights*torch.stack(pooled)).sum(0)

# Check both checkpoint compatibility and attention-grid geometry before touching DICOM data.
_probe_model=StudyCoAtNet(load_pretrained=True).to(DEVICE).eval()
with torch.inference_mode():
    _probe_x=torch.zeros((1,3,IMAGE_SIZE,IMAGE_SIZE),dtype=torch.uint8,device=DEVICE)
    _probe_plane=torch.zeros(1,dtype=torch.long,device=DEVICE)
    _probe_y=_probe_model(_probe_x,_probe_plane)
assert tuple(_probe_y.shape)==(len(LABEL_COLS),), tuple(_probe_y.shape)
del _probe_model,_probe_x,_probe_plane,_probe_y
if DEVICE.type=='cuda': torch.cuda.empty_cache()
print(f"CoAtNet checkpoint + {IMAGE_SIZE}x{IMAGE_SIZE} forward pass OK; final stage will fine-tune")


CoAtNet checkpoint + 224x224 forward pass OK; final stage will fine-tune


In [5]:
def plane_of(row):
    p=str(row.get('Anatomical_Plane','')).strip().upper()
    if p.startswith('SAG'): return 'SAGITTAL'
    if p.startswith('COR'): return 'CORONAL'
    if p.startswith('AX'): return 'AXIAL'
    return 'UNKNOWN'

def select_series(df, ids):
    df=df.copy()
    df['StudyInstanceUID']=df.StudyInstanceUID.astype(str)
    df['SeriesInstanceUID']=df.SeriesInstanceUID.astype(str)
    df=df[df.StudyInstanceUID.isin(set(map(str,ids)))].drop_duplicates(['StudyInstanceUID','SeriesInstanceUID'])
    df['plane_norm']=df.apply(plane_of,axis=1)
    for col in ('Fluid_Sensitive','Fat_Suppression'):
        if col not in df: df[col]='UNKNOWN'
        df[col]=df[col].fillna('UNKNOWN').astype(str).str.upper().str.strip()
    rows=[]
    for sid,group in df.groupby('StudyInstanceUID',sort=True):
        group=group.sort_values('SeriesInstanceUID')
        picks=[]
        for plane in ('SAGITTAL','CORONAL','AXIAL'):
            candidates=group[group.plane_norm.eq(plane)]
            if len(candidates): picks.append(candidates.iloc[0])
        for plane in ('SAGITTAL','CORONAL','AXIAL'):
            candidates=group[group.plane_norm.eq(plane)]
            used=[p for p in picks if p.plane_norm==plane]
            if len(candidates)<2 or not used: continue
            first=used[0]
            rest=candidates[candidates.SeriesInstanceUID.ne(first.SeriesInstanceUID)]
            diverse=rest[(rest.Fluid_Sensitive.ne(first.Fluid_Sensitive)) | (rest.Fat_Suppression.ne(first.Fat_Suppression))]
            if len(diverse): picks.append(diverse.iloc[0])
            elif len(rest): picks.append(rest.iloc[0])
        if not picks and len(group): picks.append(group.iloc[0])
        rows.extend([{'StudyInstanceUID':sid,'SeriesInstanceUID':str(r.SeriesInstanceUID),'plane_norm':r.plane_norm} for r in picks])
    return pd.DataFrame(rows,columns=['StudyInstanceUID','SeriesInstanceUID','plane_norm'])

train_picks=select_series(train_series,pd.concat([gold.StudyInstanceUID,pseudo.StudyInstanceUID],ignore_index=True))
test_picks=select_series(test_series,sample_submission.StudyInstanceUID)
train_picks.to_csv(OUT/f'{RUN_TAG}_series_selection.csv',index=False)
assert set(gold.StudyInstanceUID).issubset(set(train_picks.StudyInstanceUID)), 'Gold study has no metadata series'
assert set(pseudo.StudyInstanceUID).issubset(set(train_picks.StudyInstanceUID)), 'Pseudo study has no metadata series'
print('gold series',len(train_picks),'test series',len(test_picks))


gold series 8968 test series 14


In [6]:
def _read_one(path):
    try:
        d=pydicom.dcmread(str(path),force=True)
        x=np.squeeze(d.pixel_array)
        if x.ndim!=2: return None
        x=x.astype(np.float32)*float(getattr(d,'RescaleSlope',1))+float(getattr(d,'RescaleIntercept',0))
        if str(getattr(d,'PhotometricInterpretation',''))=='MONOCHROME1': x=x.max()-x
        o=np.asarray(getattr(d,'ImageOrientationPatient',[]),float)
        pos=np.asarray(getattr(d,'ImagePositionPatient',[]),float)
        g=np.nan
        if len(o)==6 and len(pos)==3:
            n=np.cross(o[:3],o[3:]); norm=np.linalg.norm(n)
            if norm>0: g=float(pos @ (n/norm))
        try: inst=float(d.InstanceNumber)
        except Exception: inst=np.nan
        return (path.name,g,inst,x)
    except Exception: return None

def resize_pad(x):
    h,w=x.shape; scale=IMAGE_SIZE/max(h,w)
    nh,nw=max(1,round(h*scale)),max(1,round(w*scale))
    z=cv2.resize(x,(nw,nh),interpolation=cv2.INTER_AREA if scale<1 else cv2.INTER_LINEAR)
    out=np.zeros((IMAGE_SIZE,IMAGE_SIZE),np.uint8)
    top=(IMAGE_SIZE-nh)//2; left=(IMAGE_SIZE-nw)//2
    out[top:top+nh,left:left+nw]=np.rint(z*255).astype(np.uint8)
    return out

def load_plane(study,series,root):
    directory=root/str(study)/str(series)
    paths=sorted(p for p in directory.iterdir() if p.is_file() and not p.name.startswith('.')) if directory.is_dir() else []
    if not paths: return None
    with ThreadPoolExecutor(max_workers=NUM_DICOM_THREADS) as pool:
        rows=[r for r in pool.map(_read_one,paths) if r is not None]
    if not rows: return None
    key=1 if all(np.isfinite(r[1]) for r in rows) else 2 if all(np.isfinite(r[2]) for r in rows) else 0
    rows.sort(key=lambda r:(r[key],r[0]) if key else (r[0],))
    samples=np.concatenate([r[3][::8,::8].ravel() for r in rows])
    samples=samples[np.isfinite(samples)]
    if not len(samples): return None
    lo,hi=np.percentile(samples,[1,99])
    n=len(rows); centers=np.rint(np.linspace(.05*(n-1),.95*(n-1),CENTERS_PER_PLANE)).astype(int)
    needed=sorted(set(np.clip(np.r_[centers-1,centers,centers+1],0,n-1)))
    maps={}
    for i in needed:
        x=np.nan_to_num(rows[i][3],nan=lo,posinf=hi,neginf=lo)
        x=(np.clip(x,lo,hi)-lo)/(hi-lo) if hi>lo else np.zeros_like(x)
        maps[i]=resize_pad(x.astype(np.float32))
    return np.stack([np.stack([maps[max(0,i-1)],maps[i],maps[min(n-1,i+1)]],0) for i in centers],0)

def load_study(study,picks,root):
    """Decode + preprocess a study in memory only. Nothing is written to disk."""
    blocks=[]; planes=[]
    group=picks[picks.StudyInstanceUID.eq(str(study))]
    for row in group.itertuples(index=False):
        try: block=load_plane(study,row.SeriesInstanceUID,root)
        except Exception: block=None
        if block is None: continue
        blocks.append(block)
        plane_id=('SAGITTAL','CORONAL','AXIAL').index(row.plane_norm) if row.plane_norm in ('SAGITTAL','CORONAL','AXIAL') else 3
        planes.extend([plane_id]*len(block))
    if not blocks: return None
    return np.concatenate(blocks,0),np.asarray(planes,np.int64)

print('Disk image caching disabled: studies will be decoded on demand.')


Disk image caching disabled: studies will be decoded on demand.


In [7]:

def first_item(batch):
    # Top-level function is safe with multiprocessing spawn/fork; lambdas are not.
    return batch[0]

def save_compact_checkpoint(model,path):
    # Frozen backbone weights already live in /kaggle/input; only save parameters that changed.
    torch.save({
        'backbone_tail': {k:v.detach().cpu() for k,v in model.backbone.stages[-1].state_dict().items()},
        'head': {k:v.detach().cpu() for k,v in model.head.state_dict().items()},
        'plane_weight': model.plane_weight.detach().cpu(),
    }, path)

def load_compact_checkpoint(model,path):
    state=torch.load(path,map_location='cpu',weights_only=True)
    model.backbone.stages[-1].load_state_dict(state['backbone_tail'],strict=True)
    model.head.load_state_dict(state['head'],strict=True)
    with torch.no_grad(): model.plane_weight.copy_(state['plane_weight'])
    return model

class BagDS(Dataset):
    def __init__(self,frame): self.frame=frame.reset_index(drop=True)
    def __len__(self): return len(self.frame)
    def __getitem__(self,i):
        row=self.frame.iloc[i]
        loaded=load_study(str(row.StudyInstanceUID),train_picks,TRAIN_SERIES_ROOT)
        if loaded is None:
            raise RuntimeError(f'Could not decode study {row.StudyInstanceUID}')
        x,plane=loaded
        return torch.from_numpy(x),torch.from_numpy(plane),torch.tensor(row[LABEL_COLS].to_numpy(np.float32))

class PseudoBagDS(BagDS):
    def __getitem__(self,i):
        x,plane,y=super().__getitem__(i)
        row=self.frame.iloc[i]
        weight=torch.tensor(row[['weight__'+c for c in LABEL_COLS]].to_numpy(np.float32))
        return x,plane,y,weight

def train_fold(model,frame,validation,fold,checkpoint):
    gold_loader=DataLoader(BagDS(frame),batch_size=1,shuffle=True,num_workers=NUM_WORKERS,
                      pin_memory=DEVICE.type=='cuda',persistent_workers=NUM_WORKERS>0,
                      collate_fn=first_item)
    print("trainable backbone parameters",sum(p.numel() for p in model.backbone.parameters() if p.requires_grad))
    optimizer=torch.optim.AdamW([{'params':[p for p in model.backbone.parameters() if p.requires_grad],'lr':LR_BACKBONE},
                                 {'params':list(model.head.parameters())+[model.plane_weight],'lr':LR_HEAD}],weight_decay=1e-4)
    scaler=torch.amp.GradScaler('cuda',enabled=DEVICE.type=='cuda')
    best_bce=np.inf; best_epoch=None; history=[]
    # The same pseudo order is shared by folds, but the model and gold split differ.
    order=np.random.default_rng(SEED).permutation(len(pseudo))
    weak_chunks=np.array_split(order,EPOCHS)
    for epoch in range(1,EPOCHS+1):
        model.train(); model.train_backbone_tail()
        losses=[]; start=time.time()
        weak_frame=pseudo.iloc[weak_chunks[epoch-1]]
        assert len(weak_frame)>0
        weak_loader=DataLoader(PseudoBagDS(weak_frame),batch_size=1,shuffle=True,num_workers=NUM_WORKERS,
                          pin_memory=DEVICE.type=='cuda',persistent_workers=False,
                          collate_fn=first_item)
        gold_iter=iter(gold_loader); weak_iter=iter(weak_loader)
        schedule=np.array([0]*len(frame)+[1]*len(weak_frame))
        np.random.default_rng(SEED+fold*100+epoch).shuffle(schedule)
        for mode in schedule:
            if mode==0:
                x,plane,y=next(gold_iter); weights=None
            else:
                x,plane,y,weights=next(weak_iter)
            x=x.to(DEVICE,non_blocking=True); plane=plane.to(DEVICE,non_blocking=True); y=y.to(DEVICE)
            optimizer.zero_grad(set_to_none=True)
            logits=model(x,plane).float()
            if mode==0:
                loss=nn.functional.binary_cross_entropy_with_logits(logits,y)
            else:
                weights=weights.to(DEVICE)
                raw=nn.functional.binary_cross_entropy_with_logits(logits,y,reduction='none')
                # Preserve uncertain/not-mentioned masking and don't inflate a study with few asserted findings.
                loss=PSEUDO_LOSS_WEIGHT*(raw*weights).sum()/weights.numel()
            if not torch.isfinite(loss): raise RuntimeError(f'Nonfinite loss in fold {fold}, epoch {epoch}')
            scaler.scale(loss).backward(); scaler.step(optimizer); scaler.update()
            losses.append(loss.item())
        pred,failed=predict_ids(model,validation.StudyInstanceUID,train_picks,TRAIN_SERIES_ROOT)
        if failed: raise RuntimeError(f'Fold {fold} validation decode failures: {failed}')
        aligned=validation[['StudyInstanceUID']].merge(pred,on='StudyInstanceUID',validate='one_to_one')
        y=validation[LABEL_COLS].to_numpy(np.float32)
        probs=aligned[LABEL_COLS].to_numpy(np.float32).clip(1e-6,1-1e-6)
        val_bce=nn.functional.binary_cross_entropy(torch.from_numpy(probs),torch.from_numpy(y)).item()
        aucs=[roc_auc_score(y[:,j],probs[:,j]) for j in range(len(LABEL_COLS)) if np.unique(y[:,j]).size==2]
        val_auc=float(np.mean(aucs)) if aucs else float('nan')
        if val_bce<best_bce-1e-6:
            best_bce=val_bce; best_epoch=epoch
            save_compact_checkpoint(model,checkpoint)
        history.append({'fold':fold,'epoch':epoch,'train_weighted_loss':float(np.mean(losses)),
                        'val_bce':val_bce,'val_macro_auc':val_auc,'n_auc_labels':len(aucs),
                        'best_so_far':best_epoch,'minutes':(time.time()-start)/60})
        print(f'fold {fold} epoch {epoch}/{EPOCHS}: weighted loss {np.mean(losses):.4f}, '
              f'val BCE {val_bce:.4f}, val AUC {val_auc:.4f} ({len(aucs)} labels), '
              f'best epoch {best_epoch}, time {(time.time()-start)/60:.1f}m')
    load_compact_checkpoint(model,checkpoint)
    return best_epoch,best_bce,history

@torch.inference_mode()
def predict_ids(model,ids,picks,root):
    model.eval(); rows=[]; failed=[]
    for sid in map(str,ids):
        loaded=load_study(sid,picks,root)
        if loaded is None:
            failed.append(sid); continue
        images,planes=loaded
        x=torch.from_numpy(images).to(DEVICE); plane=torch.from_numpy(planes).to(DEVICE)
        score=torch.sigmoid(model(x,plane).float()).cpu().numpy()
        rows.append([sid,*score.tolist()])
    return pd.DataFrame(rows,columns=['StudyInstanceUID']+LABEL_COLS),failed

folds=KFold(n_splits=N_FOLDS,shuffle=True,random_state=SEED)
oof=np.full((len(gold),len(LABEL_COLS)),np.nan,np.float32)
fold_scores=[]; epoch_history=[]
for fold,(tr,va) in enumerate(folds.split(gold),1):
    print('\nFOLD',fold,'train',len(tr),'validation',len(va))
    model=StudyCoAtNet(load_pretrained=True).to(DEVICE)
    checkpoint=OUT/f'{RUN_TAG}_fold{fold}.pt'
    best_epoch,best_bce,history=train_fold(model,gold.iloc[tr],gold.iloc[va],fold,checkpoint)
    epoch_history.extend(history)
    pred,failed=predict_ids(model,gold.iloc[va].StudyInstanceUID,train_picks,TRAIN_SERIES_ROOT)
    assert not failed
    aligned=gold.iloc[va][['StudyInstanceUID']].merge(pred,on='StudyInstanceUID',validate='one_to_one')
    oof[va]=aligned[LABEL_COLS].to_numpy(np.float32)
    fold_scores.append({'fold':fold,'n_train':len(tr),'n_validation':len(va),'checkpoint':str(checkpoint),'best_epoch':best_epoch,'best_val_bce':best_bce})
    print('saved',checkpoint)
    del model; gc.collect()
    if DEVICE.type=='cuda': torch.cuda.empty_cache()
assert np.isfinite(oof).all()
oof_df=gold[['StudyInstanceUID']+LABEL_COLS].copy()
for j,label in enumerate(LABEL_COLS): oof_df['pred__'+label]=oof[:,j]
oof_df.to_csv(OUT/f'{RUN_TAG}_oof_58.csv',index=False)
pd.DataFrame(fold_scores).to_csv(OUT/f'{RUN_TAG}_fold_manifest.csv',index=False)
pd.DataFrame(epoch_history).to_csv(OUT/f'{RUN_TAG}_epoch_history.csv',index=False)
metrics=[]
for j,label in enumerate(LABEL_COLS):
    y=gold[label].to_numpy(float)
    metrics.append({'label':label,'positives':int(y.sum()),'auc':roc_auc_score(y,oof[:,j]) if np.unique(y).size==2 else np.nan})
metrics=pd.DataFrame(metrics); metrics.to_csv(OUT/f'{RUN_TAG}_oof_auc.csv',index=False)
display(metrics); print(f'{N_FOLDS}-fold OOF macro AUC:',metrics.auc.mean())



FOLD 1 train 43 validation 15
trainable backbone parameters 12852804
fold 1 epoch 1/8: weighted loss 0.3972, val BCE 0.6765, val AUC 0.5649 (12 labels), best epoch 1, time 7.3m
fold 1 epoch 2/8: weighted loss 0.3846, val BCE 0.6409, val AUC 0.5306 (12 labels), best epoch 2, time 7.5m
fold 1 epoch 3/8: weighted loss 0.3456, val BCE 0.7051, val AUC 0.6486 (12 labels), best epoch 2, time 6.3m
fold 1 epoch 4/8: weighted loss 0.3648, val BCE 0.7611, val AUC 0.6431 (12 labels), best epoch 2, time 6.7m
fold 1 epoch 5/8: weighted loss 0.3518, val BCE 0.6002, val AUC 0.6444 (12 labels), best epoch 5, time 6.5m
fold 1 epoch 6/8: weighted loss 0.3354, val BCE 0.7881, val AUC 0.5842 (12 labels), best epoch 5, time 6.7m
fold 1 epoch 7/8: weighted loss 0.3318, val BCE 0.6779, val AUC 0.6386 (12 labels), best epoch 5, time 6.7m
fold 1 epoch 8/8: weighted loss 0.3196, val BCE 0.8682, val AUC 0.5648 (12 labels), best epoch 5, time 6.3m
saved /kaggle/working/coatnet_medgemma1750_w75_fold1.pt

FOLD 2 tr

,label,positives,auc
0,ACL,24,0.723039
1,MCL,9,0.473923
2,Medial Meniscus,26,0.746394
3,Lateral Meniscus,23,0.732919
4,Medial OA,15,0.796899
5,Lateral OA,11,0.746615
6,PF OA,21,0.763192
7,Effusion,35,0.868323
8,Synovitis,27,0.758662
9,Baker's,12,0.778986


4-fold OOF macro AUC: 0.7314637846064729


In [8]:
# No test cache is created. Each fold decodes test studies on demand.
test_ids=sample_submission.StudyInstanceUID.astype(str).tolist()
ensemble=np.zeros((len(test_ids),len(LABEL_COLS)),np.float64)
for fold in range(1,N_FOLDS+1):
    model=StudyCoAtNet(load_pretrained=True)
    load_compact_checkpoint(model,OUT/f'{RUN_TAG}_fold{fold}.pt')
    model=model.to(DEVICE)
    if DEVICE.type=='cuda': model=model.half()  # half weights and half inputs, no autocast at inference
    pred,failed=predict_ids(model,test_ids,test_picks,TEST_SERIES_ROOT)
    aligned=sample_submission[['StudyInstanceUID']].merge(pred,on='StudyInstanceUID',how='left',validate='one_to_one')
    fallback=gold[LABEL_COLS].mean().to_numpy(float)
    for j,label in enumerate(LABEL_COLS): aligned[label]=aligned[label].fillna(float(fallback[j]))
    ensemble+=aligned[LABEL_COLS].to_numpy(float)/N_FOLDS
    print('fold',fold,'predicted',len(pred),'missing',len(failed))
    del model; gc.collect()
    if DEVICE.type=='cuda': torch.cuda.empty_cache()
submission=sample_submission[['StudyInstanceUID']].copy()
submission[LABEL_COLS]=ensemble.astype(np.float32)
assert list(submission.columns)==list(sample_submission.columns)
assert submission.StudyInstanceUID.is_unique and len(submission)==len(sample_submission)
assert np.isfinite(ensemble).all() and ensemble.min()>=0 and ensemble.max()<=1
submission.to_csv(OUT/'submission.csv',index=False)
print('wrote',OUT/'submission.csv','shape',submission.shape)
display(submission.head())


fold 1 predicted 3 missing 0
fold 2 predicted 3 missing 0
fold 3 predicted 3 missing 0
fold 4 predicted 3 missing 0
wrote /kaggle/working/submission.csv shape (3, 13)


,StudyInstanceUID,ACL,MCL,Medial Meniscus,Lateral Meniscus,Medial OA,Lateral OA,PF OA,Effusion,Synovitis,Baker's,Contusion,Fracture
0,1.2.826.0.1.3680043.8.498.10047035057544427318...,0.151612,0.215540,0.520757,0.296621,0.589487,0.298977,0.577897,0.484314,0.613282,0.481488,0.514789,0.132468
1,1.2.826.0.1.3680043.8.498.10062861783145312629...,0.081616,0.159898,0.327397,0.214515,0.324566,0.201261,0.422743,0.291013,0.306597,0.241478,0.313251,0.095319
2,1.2.826.0.1.3680043.8.498.10067514707072572280...,0.097333,0.191244,0.399142,0.255320,0.441263,0.251447,0.427342,0.322611,0.306929,0.294905,0.309565,0.109245
